# First-wall workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nukehub-dev/nucleide/blob/main/notebooks/first-wall-workflow.ipynb)

End-to-end first-wall chain in one notebook: tokamak source, wall geometry, weight windows, particle exchange, activation parsing, damage, clearance screening, tritium scoping, and a spectrum check. Nucleide provides the glue; running transport or activation codes stays outside Nucleide.

> On Colab, run `%pip install "nucleide>=0.16.0"` first. Every number below is synthetic hand numbers — no evaluated nuclear data is bundled or fetched, and the wall load (1.5 MW/m2, override in the first code cell) is the one caller slot for H5a.

In [ ]:
import nucleide.plasma_source as ps

# Caller override slot: D-T wall load [MW/m2]. Valid span is 1-2.
WALL_LOAD_MW_M2 = 1.5  # synthetic default
JOULE_PER_MEV = 1.602176634e-13

# 14.1 MeV per D-T neutron -> total neutron flux [n/cm2/s].
flux_n_cm2_s = WALL_LOAD_MW_M2 * 1e6 / (14.1 * JOULE_PER_MEV) / 1e4
assert 6e13 < flux_n_cm2_s < 7e13

# Synthetic ITER-like profiles: every value is a hand number.
spec = {
    "kind": "parametric",
    "reaction": "dt",
    "major_radius": 620.0,
    "minor_radius": 200.0,
    "elongation": 1.7,
    "triangularity": 0.3,
    "shafranov_factor": 10.0,
    "mode": "H",
    "pedestal_radius": 150.0,
    "ion_density_centre": 1.0e20,
    "ion_density_peaking_factor": 1.0,
    "ion_density_pedestal": 5.0e19,
    "ion_density_separatrix": 2.0e19,
    "ion_temperature_centre": 20.0,
    "ion_temperature_peaking_factor": 1.5,
    "ion_temperature_beta": 1.5,
    "ion_temperature_pedestal": 3.0,
    "ion_temperature_separatrix": 0.5,
}
parts = ps.particles(spec, 2048, seed=7)
energies = parts["energy"]
print(f"wall load {WALL_LOAD_MW_M2} MW/m2 -> {flux_n_cm2_s:.5g} n/cm2/s")
print(f"{len(energies)} sampled, mean {float(energies.mean()):.4f} MeV")
cards = ps.emit_source_cards(spec, bins=9)
print([row["quantity"] for row in cards["sdef"]["drift"]])
assert cards["spectrum"]["mean_mev"] == 14.072874253353246

## Wall geometry and material cards

Synthetic slab stack (0.2 cm W + 0.5 cm CuCrZr + 1.3 cm EUROFER97) translated to OpenMC and Serpent, plus material-card plumbing on a synthetic two-nuclide mix. The pure-tungsten mix raises a clear error for the FLUKA dialect.

In [ ]:
import nucleide.emit as emit
import nucleide.mcnp as mcnp

fw_deck = """first wall slab stack synthetic
W CuCrZr EUROFER97 thicknesses synthetic, centimetre units
1 1 -19.3 -1
2 2 -8.9 -2
3 3 -7.8 -3
4 0 1

1 rpp 0 10 0 10 0 0.2
2 rpp 0 10 0 10 0.2 0.7
3 rpp 0 10 0 10 0.7 2.0

m1 74182 0.7 74184 0.3
m2 29100 0.2 29000 0.8
m3 26100 0.6 24000 0.4
"""
openmc_xml, openmc_drift = mcnp.parse_csg_to_openmc(fw_deck)
print(f"OpenMC {len(openmc_xml)} chars, drift {[d['action'] for d in openmc_drift]}")
serpent_text, _ = mcnp.parse_csg_to_serpent(fw_deck)
print(f"Serpent {len(serpent_text)} chars")
assert "surf 1 cuboid 0 10 0 10 0 0.2" in serpent_text.splitlines()

proxy_cards = emit.emit_cards({"U235": 5.0, "U238": 95.0}, "fw_proxy", density=19.1)
print(proxy_cards["MCNP"][:50])
try:
    emit.emit_cards({"W182": 93.0, "W184": 7.0}, "tungsten", density=19.3)
except ValueError as exc:
    print("ValueError:", exc)

## Weight windows and particle exchange

MAGIC lower bounds from the committed sample mesh tally, reformatted for OpenMC and Serpent, then 64 sampled births projected to MCPL records in temporary files.

In [ ]:
import os
import tempfile

import nucleide.mcpl as mcpl
import nucleide.vr as vr

meshtal = mcnp.read_meshtal("fixtures/mcnp/meshtal/mcnp_meshtal_single_meshtal.txt")
tally = meshtal.tallies[4]
magic_out = vr.magic(tally)
print(magic_out.groups_per_ve, len(magic_out.lower_bounds_ww))
grouped = vr.magic_with(tally, selection="per_group", tolerance=0.5, null_value=0.0)
openmc_windows = vr.emit_openmc_weight_windows(tally, grouped, mesh_id=1, window_id=1)
print(openmc_windows["notes"][0])
serpent_windows = vr.emit_serpent_wwin(tally, grouped, name="ww1", file="windows.wwd")
print(serpent_windows["card"])

tmp_dir = tempfile.mkdtemp(prefix="nucleide-fw-")
header = {
    "srcname": "fw_synth",
    "comments": [],
    "has_userflags": False,
    "has_polarisation": True,
    "double_prec": False,
    "universal_pdgcode": None,
    "universal_weight": None,
    "blobs": [],
}
mcpl_particles = []
for idx in range(64):
    mcpl_particles.append(
        {
            "ekin": float(parts["energy"][idx]),
            "polarisation": [0.0, 0.0, 0.0],
            "position": [float(parts["x"][idx]), float(parts["y"][idx]), float(parts["z"][idx])],
            "direction": [float(parts["u"][idx]), float(parts["v"][idx]), float(parts["w"][idx])],
            "time": 0.0,
            "weight": float(parts["weight"][idx]),
            "pdgcode": 2112,
            "userflags": 0,
        }
    )
first_path = os.path.join(tmp_dir, "fw1.mcpl")
mcpl.write_mcpl(first_path, header, mcpl_particles)
mcpl_file = mcpl.read_mcpl(first_path)
print(mcpl_file.nparticles, mcpl_file.srcname)
assert mcpl_file.nparticles == 64
stats = mcpl.mcpl_stats(first_path)
print(stats["nparticles"], f"{stats['ekin_mean']:.4f}", stats["pdg_counts"])

## Activation workflow from committed samples

One-year irradiation with cooling to 100 years (the multi-year tail), parsed without running ALARA or FISPACT-II.

In [ ]:
from pathlib import Path

import nucleide.alara as alara
import nucleide.fispact as fispact
import nucleide.r2s as r2s

alara_deck_text = Path("fixtures/alara/decks/sample2").read_text()
workflow = r2s.r2s_from_deck(alara_deck_text)
r2s.r2s_validate(workflow, alara_deck_text)
print(workflow["steps"], workflow["cooling_s"])
print(r2s.r2s_expand(alara_deck_text))
alara_output = Path("fixtures/alara/output/sample2.out").read_text()
source = r2s.r2s_assemble(alara_output, "sample2", "inner_zone", 4)
print(source["zone"], f"{source['total']:.6g}")
alara_rows = alara.alara_parse_output(alara_output, "sample2")
assert len(alara_rows) == 2838
clearance_rows = fispact.fispact_parse_clearance(Path("fixtures/fispact/clearance.out").read_text())
print(len(clearance_rows), sorted({r["interval"] for r in clearance_rows}))

## Damage and clearance for one full-power year

First-wall and backplate fluxes from H0 with synthetic splits and responses, plus screening of the parsed inventory against synthetic toy limits.

In [ ]:
import nucleide.damage as damage

bounds = [0.0, 0.1, 1.0, 20.0]
fw_flux = [flux_n_cm2_s * frac for frac in (0.15, 0.25, 0.60)]
bp_flux = [fw * att for fw, att in zip(fw_flux, (0.20, 0.15, 0.10), strict=False)]
dpa_xs = [100.0, 200.0, 50.0]
arc_xs = [80.0, 160.0, 40.0]
he_xs = [0.5, 1.0, 0.25]
seconds = damage.FPY_SECONDS
fw_nrt = damage.nrt_dpa(fw_flux, dpa_xs, bounds, seconds)
fw_arc = damage.arc_dpa(fw_flux, arc_xs, bounds, seconds)
bp_nrt = damage.nrt_dpa(bp_flux, dpa_xs, bounds, seconds)
fw_appm = damage.gas_appm(fw_flux, he_xs, bounds, seconds)
fw_ratio = damage.he_dpa_ratio(fw_flux, he_xs, dpa_xs, bounds, seconds)
print(f"first wall 1 fpy: NRT {fw_nrt:.5g} dpa, arc {fw_arc:.5g} dpa, He {fw_appm:.5g} appm")
print(f"backplate 1 fpy: NRT {bp_nrt:.5g} dpa")
assert fw_ratio == fw_appm / fw_nrt
mean = [0.0] * 6
cov = [[0.0004 if i == j else 0.0 for j in range(6)] for i in range(6)]
uq_out = damage.fold_uq("nrt_dpa", fw_flux, dpa_xs, bounds, seconds, mean, cov, 4000, 20260915, 5.0)
print(f"fold UQ mean {uq_out['mean']:.5g} std {uq_out['std']:.5g} passed={uq_out['passed']}")
assert uq_out["passed"]

step1_inv = {r["nuclide"]: r["activity_bq"] for r in clearance_rows if r["interval"] == 1}
toy_limits = {"V-55": 1.0, "Fe-56": 1.0, "Co-60": 4.0e6, "Rb-86m": 8.0e9}
print(alara.alara_clearance_index(step1_inv, toy_limits))
sof = alara.alara_sum_of_fractions(step1_inv, toy_limits)
print(sof["sum"], sof["class"], sof["max_nuclide"], sof["max_fraction"])
assert sof["sum"] == 2.0 and sof["class"] == "exceeded"

## Tritium scoping and foil check

Order-of-magnitude permeation through synthetic W and EUROFER stacks, then a synthetic foil unfolding check.

In [ ]:
import math

import nucleide.tritium as tritium
import nucleide.unfold as unfold

t_lag = tritium.time_lag(2e-3, 1e-9)
print(f"time lag {t_lag:.5g} s")
ss_slab = tritium.steady(
    2e-3, 32, 1e-9, {"kind": "dirichlet", "value": 1.0}, {"kind": "dirichlet", "value": 0.0}
)
print(f"slab flux {ss_slab['flux_right']:.5g} mol/m2/s")
layers = [
    {"thickness": 2e-3, "cells": 16, "D": 1e-9, "solubility": 0.1},
    {"thickness": 10e-3, "cells": 32, "D": 5e-10, "solubility": 0.1},
]
ss_stack = tritium.steady_layers(
    layers, {"kind": "dirichlet", "value": 1.0}, {"kind": "dirichlet", "value": 0.0}
)
print(f"stack flux {ss_stack['flux_right']:.5g} mol/m2/s")
assert ss_stack["flux_right"] > 0.0


def synthetic_response(n_det, n_groups, width):
    rows = []
    for i in range(n_det):
        center = i * (n_groups - 1) / max(n_det - 1, 1)
        rows.append([math.exp(-(((j - center) / width) ** 2)) + 1e-3 for j in range(n_groups)])
    return rows


def log_midpoints(n_groups, lo, hi):
    step = math.log(hi / lo) / n_groups
    return [math.exp(math.log(lo) + (j + 0.5) * step) for j in range(n_groups)]


midpoints = log_midpoints(6, 1e-6, 10.0)
truth = [en * math.exp(-en / 2.53e-5) + 1e-6 / en for en in midpoints]
response = synthetic_response(6, 6, 0.9)
rates = unfold.forward_fold(response, truth)
unfolded = unfold.sandii(
    response, rates, [3.0 * p for p in truth], tolerance=1e-12, max_iterations=10_000
)
worst = max(abs(a - b) / b for a, b in zip(unfolded["spectrum"], truth, strict=False))
print(unfolded["iterations"], f"worst error {worst:.3e}")
assert worst < 1e-9